In [ ]:
# R7_deployed_correspondence.ipynb  (revision stage 4 - deployed-booster coefficients)
# Measures the DEPLOYED depth-one booster's own per-bin curve s_j(b) and its
# projection slope onto WoE, on the SAME in-fold grids/folds as the free-fit
# correspondence (R1 for GMSC, R1b for German/HELOC), so the free-slope numbers
# reproduce the published 0.987 / 0.973 / 0.953. A depth-one booster on the WoE
# grid is additive across features, so s_j(b) is recovered by an exact additive
# decomposition (verified to machine precision) and projected onto WoE. Answers
# the open 'bin shapes and points of the deployed booster' item (Table 1).
# Single-cell; greyscale seaborn; dpi600 png+pdf; project-root-relative paths.

# R7 (faithful): deployed-booster per-bin curve s_j(b) projection slope onto WoE.
# Protocol is IDENTICAL to R1 (GMSC) and R1b (German, HELOC) so the free-fit slope
# reproduces the published correspondence (0.987 / 0.973 / 0.953); the deployed
# depth-1 booster (lr=0.3, 300 rounds, the exact prediction-track object) is fit on
# the SAME in-fold WoE grid and its s_j(b) curve is read off by an exact additive
# decomposition (verified to machine precision), then projected onto WoE.
import os, sys, time, warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED=42; IV_THRESH=0.02; SPECIAL=[-7,-8,-9]
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..'))
DATA=os.path.join(ROOT,'data')
OUT=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')

def proj_slope(scurve, woe, w):
    wv=woe-np.sum(w*woe)/w.sum(); sv=scurve-np.sum(w*scurve)/w.sum()
    den=np.sum(w*wv**2); return float(np.sum(w*sv*wv)/den) if den>0 else np.nan

def deployed_curve(gb, Xw, keep, woeb, pop):
    """Exact additive per-feature contribution s_j(b) of a depth-1 booster on the WoE grid."""
    base=Xw.mean(axis=0).copy(); f0=float(gb.decision_function(base.reshape(1,-1))[0])
    scur={}; sraw={}
    for i,c in enumerate(keep):
        probe=np.tile(base,(len(woeb[c]),1)); probe[:,i]=woeb[c]
        s=gb.decision_function(probe)-f0; sraw[c]=s
        scur[c]=s-np.sum(pop[c]*s)/pop[c].sum()
    return scur, sraw, f0

# ======================= GMSC : R1 protocol (N_SPLIT=20) =======================
def run_gmsc():
    N_SPLIT=20
    dfc=pd.read_parquet(os.path.join(DATA,'gmsc_clean.parquet')); TARGET='SeriousDlqin2yrs'
    FEATURES=[c for c in dfc.columns if c!=TARGET]
    y_all=dfc[TARGET].values.astype(int); X_all=dfc[FEATURES]
    def tree_edges(x,yv,max_leaf=6,min_frac=0.05):
        m=x.notna().values; xx=x[m].values.reshape(-1,1); yy=yv[m]
        if len(np.unique(xx))<=2: return []
        t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
        t.fit(xx,yy); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)
    def bin_counts(x,yv,edges,TG,TB,min_n):
        cuts=[-np.inf]+list(edges)+[np.inf]; idx=pd.cut(x,cuts,labels=False,include_lowest=True); recs=[]
        for b in range(len(cuts)-1):
            sel=idx==b; g=int(((yv==0)&sel).sum()); bd=int(((yv==1)&sel).sum())
            if g+bd>0: recs.append({'lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd,'miss':False})
        selm=x.isna(); gm=int(((yv==0)&selm).sum()); bm=int(((yv==1)&selm).sum())
        if gm+bm>0: recs.append({'lo':np.nan,'hi':np.nan,'good':gm,'bad':bm,'miss':True})
        for r in recs:
            gr=(r['good']+0.5)/(TG+0.5); br=(r['bad']+0.5)/(TB+0.5)
            r['woe']=float(np.log(br/gr)); r['n']=r['good']+r['bad']
            if r['miss'] and r['n']<min_n: r['woe']=0.0
        return recs
    def enforce_mono(x,yv,edges,TG,TB,min_n):
        e=list(edges)
        while e:
            recs=[r for r in bin_counts(x,yv,e,TG,TB,min_n) if not r['miss']]
            br=np.array([r['bad']/r['n'] for r in recs])
            if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): return e
            d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
        return e
    def fit_binning(Xtr,ytr):
        TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=int(0.01*len(ytr)); maps={}; iv={}
        for c in FEATURES:
            edges=enforce_mono(Xtr[c],ytr,tree_edges(Xtr[c],ytr),TG,TB,min_n)
            recs=bin_counts(Xtr[c],ytr,edges,TG,TB,min_n); maps[c]=recs
            iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in recs]))
        keep=[c for c in FEATURES if iv[c]>=IV_THRESH]; return maps,keep
    def apply_bins(x,recs):
        idx=np.full(len(x),-1,int); fin=[r for r in recs if not r['miss']]
        for p,r in enumerate(fin):
            sel=(~x.isna()).values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
        miss_pos=[p for p,r in enumerate(recs) if r['miss']]
        idx[x.isna().values]=miss_pos[0] if miss_pos else 0; idx[idx==-1]=0; return idx
    def woe_design(Xdf,maps,keep):
        return np.column_stack([np.array([r['woe'] for r in maps[c]])[apply_bins(Xdf[c],maps[c])] for c in keep])
    def dummy_design(Xdf,maps,keep):
        cols=[]; key=[]
        for c in keep:
            idx=apply_bins(Xdf[c],maps[c]); K=len(maps[c])
            for b in range(K-1): cols.append((idx==b).astype(float)); key.append((c,b))
        return np.column_stack(cols),key
    skf=StratifiedKFold(n_splits=N_SPLIT,shuffle=True,random_state=SEED)
    rows=[]; recon=[]; t0=time.time()
    for fold,(tr,te) in enumerate(skf.split(X_all,y_all)):
        Xtr=X_all.iloc[tr]; ytr=y_all[tr]; maps,keep=fit_binning(Xtr,ytr)
        pop={c:np.array([r['n'] for r in maps[c]],float) for c in keep}
        woeb={c:np.array([r['woe'] for r in maps[c]]) for c in keep}
        Xw=woe_design(Xtr,maps,keep)
        mw=sm.Logit(ytr,sm.add_constant(Xw)).fit(disp=0); beta={c:mw.params[i+1] for i,c in enumerate(keep)}
        Xd,key=dummy_design(Xtr,maps,keep)
        clf=LogisticRegression(C=5000,max_iter=4000,solver='lbfgs').fit(Xd,ytr)
        gamma={c:np.zeros(len(maps[c])) for c in keep}
        for (c,b),v in zip(key,clf.coef_.ravel()): gamma[c][b]=v
        for c in keep: gamma[c]-=np.sum(pop[c]*gamma[c])/pop[c].sum()
        gb=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,random_state=SEED+fold).fit(Xw,ytr)
        scur,sraw,f0=deployed_curve(gb,Xw,keep,woeb,pop)
        idxs={c:apply_bins(Xtr[c],maps[c]) for c in keep}
        rec=np.full(len(Xtr),f0,float)
        for c in keep: rec=rec+sraw[c][idxs[c]]
        recon.append(float(np.max(np.abs(rec-gb.decision_function(Xw)))))
        for c in keep:
            rows.append([fold,c,beta[c],proj_slope(gamma[c],woeb[c],pop[c]),proj_slope(scur[c],woeb[c],pop[c])])
    return _finish('GMSC',rows,recon,t0)

# ================== German / HELOC : R1b protocol ==================
def run_r1b(name,df,TARGET,cat_cols,num_cols,N_SPLIT):
    y=df[TARGET].values.astype(int); FEAT=cat_cols+num_cols
    def woe_of(g,b,TG,TB): return float(np.log(((b+0.5)/(TB+0.5))/((g+0.5)/(TG+0.5))))
    def tree_edges(x,yv,max_leaf=5,min_frac=0.05):
        m=~x.isin(SPECIAL)&x.notna(); xx=x[m].values.reshape(-1,1).astype(float); yy=yv[m.values]
        if len(np.unique(xx))<=2: return []
        t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
        t.fit(xx,yy); return sorted(float(v) for v in t.tree_.threshold[t.tree_.feature>=0])
    def build_maps(Xtr,ytr):
        TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=max(int(0.02*len(ytr)),10); maps={}; iv={}
        for c in cat_cols:
            recs=[]
            for lv in sorted(Xtr[c].dropna().unique()):
                sel=Xtr[c]==lv; g=int(((ytr==0)&sel.values).sum()); b=int(((ytr==1)&sel.values).sum())
                recs.append({'kind':'lvl','lv':lv,'good':g,'bad':b})
            keep=[]; og=ob=0
            for r in recs:
                if r['good']+r['bad']<min_n: og+=r['good']; ob+=r['bad']
                else: keep.append(r)
            if og+ob>0: keep.append({'kind':'other','good':og,'bad':ob})
            for r in keep: r['woe']=woe_of(r['good'],r['bad'],TG,TB); r['n']=r['good']+r['bad']
            maps[c]=keep; iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep]))
        for c in num_cols:
            edges=tree_edges(Xtr[c],ytr)
            def tbl(e):
                cuts=[-np.inf]+list(e)+[np.inf]; m=~Xtr[c].isin(SPECIAL)&Xtr[c].notna()
                idx=pd.cut(Xtr[c][m],cuts,labels=False,include_lowest=True); rr=[]
                for b in range(len(cuts)-1):
                    sel=idx==b; g=int((ytr[m.values][sel]==0).sum()); bd=int((ytr[m.values][sel]==1).sum())
                    if g+bd>0: rr.append({'kind':'num','lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd})
                return rr
            e=list(edges)
            while e:
                rr=tbl(e); br=np.array([r['bad']/(r['good']+r['bad']) for r in rr])
                if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): break
                d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
            recs=tbl(e)
            for code in SPECIAL:
                sel=Xtr[c]==code; g=int((ytr[sel.values]==0).sum()); b=int((ytr[sel.values]==1).sum())
                if g+b>0: recs.append({'kind':'code','code':code,'good':g,'bad':b})
            seln=Xtr[c].isna(); gn=int((ytr[seln.values]==0).sum()); bn=int((ytr[seln.values]==1).sum())
            if gn+bn>0: recs.append({'kind':'nan','good':gn,'bad':bn})
            keep=[]; og=ob=0
            for r in recs:
                if r['kind'] in ('code','nan') and r['good']+r['bad']<min_n: og+=r['good']; ob+=r['bad']
                else: keep.append(r)
            if og+ob>0: keep.append({'kind':'other','good':og,'bad':ob})
            for r in keep: r['woe']=woe_of(r['good'],r['bad'],TG,TB); r['n']=r['good']+r['bad']
            maps[c]=keep; iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep]))
        keep=[c for c in FEAT if iv[c]>=IV_THRESH]; return maps,keep
    def to_idx(x,recs):
        idx=np.full(len(x),-1,int)
        for p,r in enumerate(recs):
            if r['kind']=='num': sel=(~x.isin(SPECIAL)).values&x.notna().values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
            elif r['kind']=='lvl': idx[(x==r['lv']).values]=p
            elif r['kind']=='code': idx[(x==r['code']).values]=p
            elif r['kind']=='nan': idx[x.isna().values]=p
        oth=[p for p,r in enumerate(recs) if r['kind']=='other']; idx[idx==-1]=oth[0] if oth else 0; return idx
    def woe_design(Xdf,maps,keep):
        return np.column_stack([np.array([r['woe'] for r in maps[c]])[to_idx(Xdf[c],maps[c])] for c in keep])
    def dummy_design(Xdf,maps,keep):
        cols=[]; k=[]
        for c in keep:
            idx=to_idx(Xdf[c],maps[c]); K=len(maps[c])
            for b in range(K-1): cols.append((idx==b).astype(float)); k.append((c,b))
        return np.column_stack(cols),k
    skf=StratifiedKFold(n_splits=N_SPLIT,shuffle=True,random_state=SEED)
    rows=[]; recon=[]; t0=time.time()
    for fold,(tr,te) in enumerate(skf.split(df[FEAT],y)):
        Xtr=df.iloc[tr]; ytr=y[tr]; maps,keep=build_maps(Xtr,ytr)
        if len(keep)<2: continue
        pop={c:np.array([r['n'] for r in maps[c]],float) for c in keep}
        woeb={c:np.array([r['woe'] for r in maps[c]]) for c in keep}
        Xw=woe_design(Xtr,maps,keep)
        try: mw=sm.Logit(ytr,sm.add_constant(Xw)).fit(disp=0)
        except Exception: continue
        beta={c:mw.params[i+1] for i,c in enumerate(keep)}
        Xd,key=dummy_design(Xtr,maps,keep)
        clf=LogisticRegression(C=5000,max_iter=5000).fit(Xd,ytr)
        gamma={c:np.zeros(len(maps[c])) for c in keep}
        for (c,b),v in zip(key,clf.coef_.ravel()): gamma[c][b]=v
        for c in keep: gamma[c]-=np.sum(pop[c]*gamma[c])/pop[c].sum()
        gb=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,random_state=SEED+fold).fit(Xw,ytr)
        scur,sraw,f0=deployed_curve(gb,Xw,keep,woeb,pop)
        idxs={c:to_idx(Xtr[c],maps[c]) for c in keep}
        rec=np.full(len(Xtr),f0,float)
        for c in keep: rec=rec+sraw[c][idxs[c]]
        recon.append(float(np.max(np.abs(rec-gb.decision_function(Xw)))))
        for c in keep:
            rows.append([fold,c,beta[c],proj_slope(gamma[c],woeb[c],pop[c]),proj_slope(scur[c],woeb[c],pop[c])])
    return _finish(name,rows,recon,t0)

def _finish(name,rows,recon,t0):
    R=pd.DataFrame(rows,columns=['fold','feature','beta_hat','slope_free','slope_deploy'])
    agg=R.groupby('feature').agg(beta=('beta_hat','mean'),beta_sd=('beta_hat','std'),
            slope_free=('slope_free','mean'),slope_free_sd=('slope_free','std'),
            slope_deploy=('slope_deploy','mean'),slope_deploy_sd=('slope_deploy','std')).reset_index()
    agg['gap_deploy']=(agg['beta']-agg['slope_deploy']).abs(); agg['gap_free']=(agg['beta']-agg['slope_free']).abs()
    r_dep=np.corrcoef(agg['beta'],agg['slope_deploy'])[0,1]
    r_free=np.corrcoef(agg['beta'],agg['slope_free'])[0,1]
    r_fd=np.corrcoef(agg['slope_free'],agg['slope_deploy'])[0,1]
    summ=dict(dataset=name,n_feat=len(agg),folds=R['fold'].nunique(),
              corr_beta_deploy=r_dep,corr_beta_free=r_free,corr_free_deploy=r_fd,
              mean_gap_deploy=agg['gap_deploy'].mean(),mean_gap_free=agg['gap_free'].mean(),
              max_recon_err=float(np.max(recon)))
    print('\n===== %s (%d folds, %.0fs) ====='%(name,summ['folds'],time.time()-t0))
    print(agg.round(4).to_string(index=False))
    print('corr(beta,deploy)=%.4f | corr(beta,free)=%.4f | corr(free,deploy)=%.4f | mean|gap|dep=%.4f free=%.4f | recon=%.1e'%(
        r_dep,r_free,r_fd,summ['mean_gap_deploy'],summ['mean_gap_free'],summ['max_recon_err']))
    R.round(6).to_csv(os.path.join(OUT,'R7_%s_slope_long.csv'%name),index=False)
    agg.round(6).to_csv(os.path.join(OUT,'R7_%s_slope_agg.csv'%name),index=False)
    return summ

summaries=[run_gmsc()]
cols=['Status_checking','Duration','Credit_history','Purpose','Credit_amount','Savings','Employment_since',
      'Installment_rate','Personal_status_sex','Other_debtors','Residence_since','Property','Age',
      'Other_installment','Housing','Existing_credits','Job','Num_liable','Telephone','Foreign_worker','target']
gm=pd.read_csv(os.path.join(DATA,'german.data'),sep=' ',header=None,names=cols); gm['target']=(gm['target']==2).astype(int)
gmcat=['Status_checking','Credit_history','Purpose','Savings','Employment_since','Personal_status_sex',
       'Other_debtors','Property','Other_installment','Housing','Job','Telephone','Foreign_worker']
gmnum=['Duration','Credit_amount','Installment_rate','Residence_since','Age','Existing_credits','Num_liable']
summaries.append(run_r1b('German',gm,'target',gmcat,gmnum,5))
h=pd.read_csv(os.path.join(DATA,'heloc.csv')); h['RiskPerformance']=(h['RiskPerformance']=='Bad').astype(int)
hf=[c for c in h.columns if c!='RiskPerformance']; allmiss=(h[hf]==-9).all(axis=1); h=h[~allmiss].reset_index(drop=True)
summaries.append(run_r1b('HELOC',h,'RiskPerformance',[],hf,10))
S=pd.DataFrame(summaries); S.to_csv(os.path.join(OUT,'R7_deployed_slope_summary.csv'),index=False)
print('\n================ R7 SUMMARY ================'); print(S.round(4).to_string(index=False))


# ================= FIGURE =================
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})

summ=pd.read_csv(os.path.join(OUT,'R7_deployed_slope_summary.csv')).set_index('dataset')
names=['GMSC','German','HELOC']
fig,axes=plt.subplots(1,3,figsize=(13.2,4.5))
for ax,nm in zip(axes,names):
    a=pd.read_csv(os.path.join(OUT,'R7_%s_slope_agg.csv'%nm))
    lo=min(a['beta'].min(),a['slope_deploy'].min(),a['slope_free'].min())
    hi=max(a['beta'].max(),a['slope_deploy'].max(),a['slope_free'].max())
    pad=0.1*(hi-lo); lo-=pad; hi+=pad
    ax.plot([lo,hi],[lo,hi],color='0.0',lw=1,ls='--',label='identity')
    ax.axhline(1.0,color='0.6',lw=1,ls=':',label='naive-Bayes = 1')
    ax.scatter(a['beta'],a['slope_free'],s=55,marker='^',facecolors='none',
               edgecolors='0.55',linewidths=1.1,label='free dummy-logit $\\hat\\gamma_j$')
    ax.errorbar(a['beta'],a['slope_deploy'],yerr=a['slope_deploy_sd'],fmt='o',ms=6,
                color='0.15',ecolor='0.6',capsize=2,label='deployed booster $s_j(b)$')
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel(r'logistic-WoE coefficient $\hat\beta_j$')
    if nm=='GMSC': ax.set_ylabel(r'projection slope onto WoE')
    ax.set_title('%s: deploy r=%.2f (free r=%.2f)'%(
        nm,summ.loc[nm,'corr_beta_deploy'],summ.loc[nm,'corr_beta_free']),fontsize=10)
    if nm=='GMSC': ax.legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=8,loc='upper left')
fig.tight_layout()
for e in ('png','pdf'):
    fig.savefig(os.path.join(FIGS,'R7_deployed_slope.'+e),dpi=600,bbox_inches='tight')
plt.close(fig)
print('Saved figure: R7_deployed_slope.{png,pdf} ->',FIGS)
